# CST - Kontinuumselemente 

In [1]:
import numpy 
import sympy 

XI = numpy.asarray(sympy.symbols('xi:' + str(2))) # symbolischer vektor mit lokalen koordinaten

a, b  = sympy.symbols('a, b') # seitenlänge des elements

"""     
           (0)___a____(1)
            |         /
 CST:       |       /
           b|     /                         KS:        __ x (i)
            |   /                                     |  
            | /                                       z
           (2)                                       (ii)

Es gibt zwei translatorische Freiheitsgrade in jedem Knoten => 6 DoFs. 
Der erste DoF ist die x-Verschiebung im Knoten (0), der zweite DoF ist 
die z-Verschiebung im Knoten (0) usw.

"""

def interpolationFunctions(XI, a, b): 

        h = sympy.ones(3,1) # initialisiere symbolischen spaltenvektor für ansatzfunktionen
         
        # ansatzfunktionen: eine für jeden knoten
        h[0] =  1 - XI[0]/a - XI[1]/b   
        h[1] =  XI[0]/a
        h[2] =  XI[1]/b

        return h
    
h = interpolationFunctions(XI, a, b)  


from IPython.display import display
#sympy.init_printing()
print("lokale Koordinaten:")
print("-------------------")    
for i in range(len(XI)):
    display(XI[i])
print("Ansatzfunktionen:")
print("-----------------")    
display(h)

lokale Koordinaten:
-------------------


xi0

xi1

Ansatzfunktionen:
-----------------


Matrix([
[1 - xi1/b - xi0/a],
[            xi0/a],
[            xi1/b]])

In [2]:
def interpolationFunctionsDerivatives(XI, h):
        """ interpolationfunction[node, derivativeaxis] """
        dh_dX = sympy.ones(3, 2) 
        for index in range(3): 
            for axis in range(2):
                dh_dX[index,axis] = sympy.diff(h[index], XI[axis]) 
        return dh_dX 
    
dh_dx = interpolationFunctionsDerivatives(XI, h)

print("Ableitungen der Ansatzfunktionen:")
print("---------------------------------")    
display(dh_dx)

Ableitungen der Ansatzfunktionen:
---------------------------------


Matrix([
[-1/a, -1/b],
[ 1/a,    0],
[   0,  1/b]])

In [3]:
X = numpy.asarray(sympy.symbols('x:' + str(2)))

u, v, u_x, u_z, v_x, v_z = sympy.symbols('u, v, u_x, u_z, v_x, v_z')

# parameters
E     = sympy.Symbol('E')
nu    = sympy.Symbol('nu') 

lmbda = (E*nu)/((1+nu)*(1-2*nu))
mu    = E/(2*(1+nu))

u     = numpy.array([u, v])                   # u (displacement field)
du_dx = numpy.array([[u_x, u_z], [v_x, v_z]]) # grad(u)


def epsilon(grad_u): 
    """ returns strain tensor """
    return 0.5 * ( grad_u + grad_u.T )

def sigma(epsilon): 
    """ computes stress tensor from strain tensor """ 
    return lmbda*numpy.trace(epsilon)*numpy.eye(2) + 2*mu*epsilon 
     

# strain energy density
W = numpy.trace(sigma(epsilon(du_dx))@epsilon(du_dx))


print(" Verzerrungsenergiedichte :")
print("---------------------------")
display(W)

 Verzerrungsenergiedichte :
---------------------------


4*E*(0.25*(u_z + v_x)**2)/(2*nu + 2) + 1.0*u_x*(1.0*E*nu*(1.0*u_x + 1.0*v_z)/((1 - 2*nu)*(nu + 1)) + 2.0*E*u_x/(2*nu + 2)) + 1.0*v_z*(1.0*E*nu*(1.0*u_x + 1.0*v_z)/((1 - 2*nu)*(nu + 1)) + 2.0*E*v_z/(2*nu + 2))

In [4]:
U_ = numpy.asarray(sympy.symbols('U:' + str(6)))

W = W.subs([ (u_x, (U_[0] * dh_dx[0,0] + U_[2] * dh_dx[1,0] + U_[4] * dh_dx[2,0]) ),      
             (v_x, (U_[1] * dh_dx[0,0] + U_[3] * dh_dx[1,0] + U_[5] * dh_dx[2,0]) ),                                           
             (u_z, (U_[0] * dh_dx[0,1] + U_[2] * dh_dx[1,1] + U_[4] * dh_dx[2,1]) ),     
             (v_z, (U_[1] * dh_dx[0,1] + U_[3] * dh_dx[1,1] + U_[5] * dh_dx[2,1]) ) ])
 

print(" Verzerrungsenergiedichte in Abhängigkeit der DoFs:")
print("---------------------------------------------------")
display(W)    

 Verzerrungsenergiedichte in Abhängigkeit der DoFs:
---------------------------------------------------


1.0*E*(-U0/b - U1/a + U3/a + U4/b)**2/(2*nu + 2) + 1.0*(-U0/a + U2/a)*(1.0*E*nu*(-1.0*U0/a - 1.0*U1/b + 1.0*U2/a + 1.0*U5/b)/((1 - 2*nu)*(nu + 1)) + 2.0*E*(-U0/a + U2/a)/(2*nu + 2)) + 1.0*(-U1/b + U5/b)*(1.0*E*nu*(-1.0*U0/a - 1.0*U1/b + 1.0*U2/a + 1.0*U5/b)/((1 - 2*nu)*(nu + 1)) + 2.0*E*(-U1/b + U5/b)/(2*nu + 2))

In [5]:
# INTEGRIEREN nach x von 0 bis (a-z) und dann nach z von 0 bis b 
W = sympy.integrate(W, ( XI[0], 0, (a-XI[1]) ))
W = sympy.integrate(W, ( XI[1], 0,  b        )) 
display(W)

b**2*(-32.0*E*U0**2*a**2*nu + 16.0*E*U0**2*a**2 - 32.0*E*U0**2*b**2*nu + 32.0*E*U0**2*b**2 + 32.0*E*U0*U1*a*b + 64.0*E*U0*U2*b**2*nu - 64.0*E*U0*U2*b**2 + 64.0*E*U0*U3*a*b*nu - 32.0*E*U0*U3*a*b + 64.0*E*U0*U4*a**2*nu - 32.0*E*U0*U4*a**2 - 64.0*E*U0*U5*a*b*nu - 32.0*E*U1**2*a**2*nu + 32.0*E*U1**2*a**2 - 32.0*E*U1**2*b**2*nu + 16.0*E*U1**2*b**2 - 64.0*E*U1*U2*a*b*nu + 64.0*E*U1*U3*b**2*nu - 32.0*E*U1*U3*b**2 + 64.0*E*U1*U4*a*b*nu - 32.0*E*U1*U4*a*b + 64.0*E*U1*U5*a**2*nu - 64.0*E*U1*U5*a**2 - 32.0*E*U2**2*b**2*nu + 32.0*E*U2**2*b**2 + 64.0*E*U2*U5*a*b*nu - 32.0*E*U3**2*b**2*nu + 16.0*E*U3**2*b**2 - 64.0*E*U3*U4*a*b*nu + 32.0*E*U3*U4*a*b - 32.0*E*U4**2*a**2*nu + 16.0*E*U4**2*a**2 - 32.0*E*U5**2*a**2*nu + 32.0*E*U5**2*a**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(32.0*E*U0**2*a**2*nu - 16.0*E*U0**2*a**2 + 32.0*E*U0**2*b**2*nu - 32.0*E*U0**2*b**2 - 32.0*E*U0*U1*a*b - 64.0*E*U0*U2*b**2*nu + 64.0*E*U0*U2*b**2 - 64.0*E*U0*U3*a*b*nu + 32.0*E*U0*U3*a*b - 64.0*E*U0*U4*a*

In [6]:
esm = sympy.zeros(6,6)
for i in range(6): 
    for j in range(6): 
        esm[i,j] = sympy.diff( sympy.diff(W, U_[i]), U_[j] )

print(" ESM: ")
print("------")
display(esm)

 ESM: 
------


Matrix([
[b**2*(-64.0*E*a**2*nu + 32.0*E*a**2 - 64.0*E*b**2*nu + 64.0*E*b**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(64.0*E*a**2*nu - 32.0*E*a**2 + 64.0*E*b**2*nu - 64.0*E*b**2)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2),                                                                                                         32.0*E*a*b**3/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) - 32.0*E*a*b**2/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2), b**2*(64.0*E*b**2*nu - 64.0*E*b**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*b**2*nu + 64.0*E*b**2)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2),     b**2*(64.0*E*a*b*nu - 32.0*E*a*b)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*a*b*nu + 32.0*E*a*b)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2), b**2*(64.0*E*a**2*nu - 32.0*E*a**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*a**2*nu + 32.0*E

In [7]:
ESM = numpy.asarray(esm)
for index in numpy.ndindex(6,6): 
    print("k_e_local",index,"=",ESM[index])

k_e_local (0, 0) = b**2*(-64.0*E*a**2*nu + 32.0*E*a**2 - 64.0*E*b**2*nu + 64.0*E*b**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(64.0*E*a**2*nu - 32.0*E*a**2 + 64.0*E*b**2*nu - 64.0*E*b**2)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2)
k_e_local (0, 1) = 32.0*E*a*b**3/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) - 32.0*E*a*b**2/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2)
k_e_local (0, 2) = b**2*(64.0*E*b**2*nu - 64.0*E*b**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*b**2*nu + 64.0*E*b**2)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2)
k_e_local (0, 3) = b**2*(64.0*E*a*b*nu - 32.0*E*a*b)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*a*b*nu + 32.0*E*a*b)/(64.0*a*b**2*nu**2 + 32.0*a*b**2*nu - 32.0*a*b**2)
k_e_local (0, 4) = b**2*(64.0*E*a**2*nu - 32.0*E*a**2)/(128.0*a**2*b**2*nu**2 + 64.0*a**2*b**2*nu - 64.0*a**2*b**2) + b*(-64.0*E*a**2*nu + 32.0*E*a**2)/(64.0*a*b**2*nu**2 +